# 📊 Visualisation et dashboard

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 2 · 9h45** · Exercice

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Construire des visualisations analytiques et communicantes
- Monter un dashboard interactif Plotly
- Y intégrer la représentativité des segments et des alertes sur les biais

In [1]:
%pip install -q fairlearn
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

✅ Configuration prête : chaque fichier de données se télécharge au premier usage


In [2]:
# Secteur traité : Crédit Azur (banque) — autres choix : "commerce", "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Crédit Azur · 20477 lignes, 20 colonnes


In [3]:
import plotly.express as px
import plotly.graph_objects as go
from config_projet import exporter_html, PALETTE
from fairlearn.metrics import MetricFrame, selection_rate, count
df = nettoyage_minimal(brut, cfg)
for col in cfg["sensibles"]:
    df[col] = df[col].fillna("inconnu")
mensuel = df.set_index(cfg["date"])[cfg["ca"]].resample("MS").sum()

## 1. Les chiffres clés
Trois indicateurs `go.Indicator` : chiffre d'affaires de 2025 avec l'évolution par rapport à 2024, part de votre segment principal, taux de décision favorable.

In [4]:
from plotly.subplots import make_subplots
annee = df[cfg["date"]].dt.year
ca25, ca24 = df.loc[annee == 2025, cfg["ca"]].sum(), df.loc[annee == 2024, cfg["ca"]].sum()
seg_ca = df.groupby(cfg["segment"])[cfg["ca"]].sum()
seg_principal = seg_ca.idxmax()
part_seg = seg_ca.max() / seg_ca.sum()
taux_global = df[cfg["decision"]].mean()
taux_24, taux_25 = df.loc[annee == 2024, cfg["decision"]].mean(), df.loc[annee == 2025, cfg["decision"]].mean()

kpis = make_subplots(rows=1, cols=3, specs=[[{"type": "indicator"}] * 3])
kpis.add_trace(go.Indicator(mode="number+delta", value=ca25 / 1e6,
                            number={"suffix": " M€", "valueformat": ".2f"},
                            delta={"reference": ca24 / 1e6, "relative": True, "valueformat": ".1%"},
                            title={"text": "PNB estimé 2025<br><span style='font-size:0.8em'>vs 2024</span>"}), row=1, col=1)
kpis.add_trace(go.Indicator(mode="number", value=part_seg * 100, number={"suffix": " %", "valueformat": ".0f"},
                            title={"text": f"Part du PNB · crédit « {seg_principal} »<br><span style='font-size:0.8em'>segment principal</span>"}), row=1, col=2)
kpis.add_trace(go.Indicator(mode="number+delta", value=taux_25 * 100, number={"suffix": " %", "valueformat": ".1f"},
                            delta={"reference": taux_24 * 100, "valueformat": ".1f", "suffix": " pts"},
                            title={"text": f"Taux d'accord 2025<br><span style='font-size:0.8em'>global : {taux_global:.1%}</span>"}), row=1, col=3)
kpis.update_layout(height=230, margin=dict(t=70, b=10, l=20, r=20), paper_bgcolor="white", font=dict(size=14))
kpis.show()

## 2. La tendance, pour les décideurs
Courbe mensuelle, tendance sur 12 mois, et l'événement du brief annoté. Le titre dit la conclusion.

In [5]:
ev = pd.Timestamp(cfg["evenement"][0])
mm12 = mensuel.rolling(12, center=True).mean()
avant, apres = mensuel[mensuel.index < ev].mean(), mensuel[mensuel.index >= ev].mean()
courbe = go.Figure()
courbe.add_trace(go.Scatter(x=mensuel.index, y=mensuel, name="PNB mensuel", mode="lines+markers",
                            line=dict(color="#BBBBBB", width=2), marker=dict(size=5)))
courbe.add_trace(go.Scatter(x=mensuel.index, y=mm12, name="Tendance (12 mois)", mode="lines",
                            line=dict(color=PALETTE[0], width=4)))
courbe.add_shape(type="line", x0=ev, x1=ev, y0=0, y1=1, yref="paper", line=dict(color=PALETTE[5], dash="dash", width=2))
courbe.add_annotation(x=ev, y=1, yref="paper", text=f"{ev:%m/%Y} · {cfg['evenement'][1]}", showarrow=False,
                      xanchor="left", yanchor="bottom", font=dict(color=PALETTE[5], size=13))
courbe.update_layout(title=f"Le PNB mensuel a perdu {1 - apres / avant:.0%} depuis le déploiement du score V2 "
                           f"({avant / 1e3:,.0f} k€ → {apres / 1e3:,.0f} k€ par mois)",
                     yaxis_title=cfg["ca_label"], xaxis_title="", template="simple_white", height=420,
                     legend=dict(orientation="h", y=-0.15), font=dict(size=13))
courbe.show()

## 3. L'analyse, pour l'équipe data
Un graphique plus riche : chiffre d'affaires mensuel par segment, avec un sélecteur interactif.

In [6]:
par_mois_seg = df.groupby([pd.Grouper(key=cfg["date"], freq="MS"), cfg["segment"]])[cfg["ca"]].sum().unstack()
segments = list(par_mois_seg.columns)
detail = go.Figure()
for i, s in enumerate(segments):
    detail.add_trace(go.Scatter(x=par_mois_seg.index, y=par_mois_seg[s], name=s, mode="lines+markers",
                                line=dict(color=PALETTE[i % len(PALETTE)], width=2.5), marker=dict(size=5)))
detail.add_trace(go.Scatter(x=par_mois_seg.index, y=par_mois_seg.sum(axis=1), name="Tous les types", mode="lines",
                            line=dict(color="black", width=2, dash="dot"), visible=True))
n = len(segments) + 1
boutons = [dict(label="Tous les types", method="update", args=[{"visible": [True] * n}])]
for i, s in enumerate(segments):
    boutons.append(dict(label=s, method="update", args=[{"visible": [j == i for j in range(n)]}]))
detail.update_layout(
    title="PNB mensuel par type de crédit : tous les types reculent après le score V2",
    updatemenus=[dict(buttons=boutons, direction="down", x=0.0, y=1.18, xanchor="left")],
    yaxis_title=cfg["ca_label"], xaxis_title="", template="simple_white", height=450, font=dict(size=13),
    legend=dict(orientation="h", y=-0.15))
detail.show()

## 4. L'indicateur de représentativité
Pour chaque variable sensible, le ratio part observée ÷ part de référence, coloré selon le statut, avec les seuils 0,8 et 1,25.

In [7]:
clients = df.drop_duplicates("client_id")
ref = reference()
lignes = []
for col in cfg["sensibles"]:
    obs = clients[col].value_counts(normalize=True).rename_axis("modalite").reset_index(name="part_observee")
    obs["variable"] = col
    lignes.append(obs)
representativite = pd.concat(lignes).merge(ref, on=["variable", "modalite"], how="left")
representativite["ratio"] = representativite["part_observee"] / representativite["part_reference"]
rep = representativite.dropna(subset=["ratio"]).copy()
rep["statut"] = np.select([rep["ratio"] < 0.8, rep["ratio"] > 1.25], ["sous-représenté", "sur-représenté"], default="conforme")
COULEURS_STATUT = {"sous-représenté": "#D55E00", "conforme": "#999999", "sur-représenté": "#0072B2"}

variables = list(rep["variable"].unique())
fig_rep = make_subplots(rows=1, cols=len(variables), subplot_titles=variables, horizontal_spacing=0.07)
for j, v in enumerate(variables, 1):
    sous = rep[rep["variable"] == v]
    fig_rep.add_trace(go.Bar(x=[f"{m}<br>{('⚠ ' + st) if st != 'conforme' else st}" for m, st in zip(sous["modalite"], sous["statut"])],
                             y=sous["ratio"], marker_color=[COULEURS_STATUT[s] for s in sous["statut"]],
                             text=[f"{r:.2f}" for r in sous["ratio"]], textposition="outside", textfont=dict(size=13),
                             showlegend=False, hovertemplate="%{x}<br>ratio %{y:.2f}<extra></extra>"), row=1, col=j)
    for seuil_r in (0.8, 1.25):
        fig_rep.add_shape(type="line", xref=f"x{j if j > 1 else ''} domain", x0=0, x1=1, yref=f"y{j if j > 1 else ''}", y0=seuil_r, y1=seuil_r,
                          line=dict(color="black", dash="dash", width=1.5))
n_sous = (rep["statut"] == "sous-représenté").sum(); n_sur = (rep["statut"] == "sur-représenté").sum()
fig_rep.update_yaxes(range=[0, max(1.6, rep["ratio"].max() * 1.25)])
fig_rep.update_layout(title=f"{n_sous} groupes sous-représentés et {n_sur} sur-représentés parmi les clients de Crédit Azur "
                            "(seuils 0,8 et 1,25)", template="simple_white", height=500, font=dict(size=12))
fig_rep.update_xaxes(tickfont=dict(size=10))
fig_rep.update_layout(uniformtext=dict(minsize=11, mode="show"))
fig_rep.show()

## 5. La parité de la décision
Taux de décision favorable par groupe, avec la ligne des quatre cinquièmes du meilleur taux.

In [8]:
y = df[cfg["decision"]]
parite = make_subplots(rows=1, cols=len(cfg["sensibles"]), subplot_titles=cfg["sensibles"], horizontal_spacing=0.07)
seuils = {}
for j, col in enumerate(cfg["sensibles"], 1):
    mf = MetricFrame(metrics={"taux": selection_rate, "effectif": count}, y_true=y, y_pred=y, sensitive_features=df[col])
    t = mf.by_group
    ordre = [a for a in ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"] if a in t.index] if col == "tranche_age" else list(t.index)
    t = t.loc[ordre]
    seuil = 0.8 * t["taux"].max(); seuils[col] = seuil
    sous = t["taux"] < seuil
    parite.add_trace(go.Bar(x=t.index, y=t["taux"], marker_color=["#D55E00" if s else "#0072B2" for s in sous],
                            text=[f"{v:.0%}<br>n={int(n)}" + ("<br>⚠ sous 4/5" if s else "") for v, n, s in zip(t["taux"], t["effectif"], sous)],
                            textposition="outside", textfont=dict(size=12), showlegend=False,
                            hovertemplate="%{x}<br>taux %{y:.1%}<extra></extra>"), row=1, col=j)
    parite.add_shape(type="line", xref=f"x{j if j > 1 else ''} domain", x0=0, x1=1, yref=f"y{j if j > 1 else ''}", y0=seuil, y1=seuil,
                     line=dict(color="black", dash="dash", width=1.5))
    parite.layout.annotations[j - 1].text = f"{col} · pointillé = 4/5 du meilleur taux ({seuil:.0%})"
parite.update_yaxes(range=[0, 1.2], tickformat=".0%", tickvals=[0, .2, .4, .6, .8, 1])
parite.update_layout(title="Les 18-24 ans et les 25-34 ans sont sous la ligne des quatre cinquièmes pour l'accord de crédit",
                     template="simple_white", height=480, font=dict(size=12))
parite.update_layout(uniformtext=dict(minsize=10, mode="show"))
parite.update_annotations(font_size=12)
parite.show()

## 6. Les alertes
Un bloc HTML qui liste en clair les groupes sous-représentés et les groupes sous la ligne des 4/5. Ne comptez pas sur la couleur seule : écrivez-le.

In [9]:
alertes_rep = rep[rep["statut"] != "conforme"].sort_values("ratio")
items = []
for _, r in alertes_rep.iterrows():
    items.append(f"<li><b>{r['variable']} = {r['modalite']}</b> : {r['statut']} "
                 f"({r['part_observee']:.0%} des clients contre {r['part_reference']:.0%} dans la population, ratio {r['ratio']:.2f}).</li>")
for col in cfg["sensibles"]:
    mf = MetricFrame(metrics={"taux": selection_rate, "effectif": count}, y_true=y, y_pred=y, sensitive_features=df[col])
    t = mf.by_group; seuil = 0.8 * t["taux"].max()
    for g, r in t[t["taux"] < seuil].sort_values("taux").iterrows():
        items.append(f"<li><b>{col} = {g}</b> : taux d'accord de {r['taux']:.0%} (n = {int(r['effectif'])}), "
                     f"sous la ligne des 4/5 ({seuil:.0%}), soit {r['taux'] / t['taux'].max():.2f} du meilleur groupe.</li>")
alertes = ('<section class="alerte"><h2>⚠ Alertes sur les biais</h2>'
           '<p class="desc">Écrites en clair : elles ne dépendent pas de la couleur. Une disparité est un signal à instruire, pas une preuve de discrimination.</p>'
           f'<ul>{"".join(items)}</ul></section>')
from IPython.display import HTML
HTML(alertes)

## 7. L'export HTML autonome
Assemblez le tout avec `exporter_html` (fourni) : une seule page qui suit le fil du discours (contexte, tendance, segments, biais, recommandations, décision), avec une barre de navigation en haut pour passer d'une étape à l'autre. Chaque figure reçoit une description courte, utile aux lecteurs d'écran.

In [10]:
ev = pd.Timestamp(cfg["evenement"][0])
dec = cfg["decision"]
fmt = lambda x: f"{x:,.0f}".replace(",", " ")            # 1 234 567 → espace fine insécable

# Chiffres réutilisés dans les blocs de texte (calculés, pas écrits à la main)
m_av, m_ap = mensuel[mensuel.index < ev].mean(), mensuel[mensuel.index >= ev].mean()
av, ap = df[df[cfg["date"]] < ev], df[df[cfg["date"]] >= ev]
t_av, t_ap = av[dec].mean(), ap[dec].mean()
gain_68 = m_ap / (t_ap * 100) * (68 - t_ap * 100)             # PNB mensuel gagné si le taux d'accord remonte à 68 %
age_av, age_ap = av.groupby("tranche_age")[dec].mean(), ap.groupby("tranche_age")[dec].mean()
ratio_av, ratio_ap = age_av.min() / age_av.max(), age_ap.min() / age_ap.max()
jeunes_acc = df[(df["tranche_age"] == "18-24") & (df[dec] == 1)]
rembourse_j = jeunes_acc[cfg["resultat"]].mean()
cl = df.drop_duplicates("client_id")
part65, tx65 = (cl["tranche_age"] == "65+").mean(), df[df["tranche_age"] == "65+"][dec].mean()

def etape(num, ancre, titre, question):
    return (f'<h2 id="{ancre}" style="font-size:26px;margin:40px 0 2px;color:#1B212C;border-bottom:3px solid #D55E00;'
            f'padding-bottom:4px;scroll-margin-top:60px">{num} · {titre}</h2>'
            f'<p style="margin:6px 0 0;color:#4E5567;font-size:16px">{question}</p>')

PLAN = [("contexte", "1 · Contexte"), ("tendance", "2 · Tendance"), ("segments", "3 · Segments"),
        ("biais", "4 · Biais"), ("recommandations", "5 · Recommandations"), ("decision", "6 · Décision")]
nav = ('<nav style="position:sticky;top:0;z-index:10;background:#fff;border-bottom:1px solid #ddd;padding:10px 0;'
       'text-align:center;font-size:15px">' +
       "".join(f'<a href="#{a}" style="margin:0 12px;color:#0072B2;font-weight:bold;text-decoration:none">{t}</a>' for a, t in PLAN) + "</nav>")

contexte = ('<section class="ok"><h2>La question</h2><ul style="font-size:17px;line-height:1.7;margin:6px 0">'
            f'<li><b>Qui :</b> {cfg["nom"]}, une banque qui prête de l\'argent (auto, travaux, personnel, renouvelable).</li>'
            f'<li><b>Ce qui change :</b> en {ev:%m/%Y}, {cfg["evenement"][1]} : une note automatique décide désormais de l\'accord du crédit.</li>'
            '<li><b>Ce qui inquiète :</b> la banque prête moins, donc gagne moins, et les jeunes actifs se plaignent.</li>'
            f'<li><b>Notre étude :</b> {fmt(len(df))} demandes de crédit, de {df[cfg["date"]].min():%m/%Y} à {df[cfg["date"]].max():%m/%Y}.</li>'
            '</ul></section>')

def tableau(entetes, lignes):
    th = "".join(f'<th style="background:#1B212C;color:#fff;text-align:left;padding:8px;font-size:14px">{h}</th>' for h in entetes)
    tr = "".join("<tr>" + "".join(f'<td style="border:1px solid #ddd;padding:8px;font-size:14px;vertical-align:top">{c}</td>' for c in l) + "</tr>" for l in lignes)
    return f'<section><table style="border-collapse:collapse;width:100%"><tr>{th}</tr>{tr}</table></section>'

reco = tableau(["Recommandation", "Constat chiffré", "Indicateur de suivi", "Limite"], [
    ["<b>1. Ajuster le seuil du score</b> : test sur 10 % des dossiers limites, pendant 2 trimestres",
     f"Taux d'accord {t_av:.0%} → {t_ap:.0%}. Remonter à 68 % : environ {fmt(round(gain_68, -3))} € de PNB en plus par mois",
     "Taux d'accord et remboursement (au moins 97 %), chaque mois", "Le PNB n'est pas le bénéfice : le coût des impayés n'est pas chiffré"],
    ["<b>2. Vérifier le score pour les moins de 35 ans</b>, relecture humaine des dossiers limites",
     f"18-24 ans : {age_av['18-24']:.0%} → {age_ap['18-24']:.0%} d'accord. Ratio de parité d'âge {ratio_av:.2f} → {ratio_ap:.2f} (seuil 0,8)",
     "Ratio de parité d'âge, chaque trimestre (cible : plus de 0,5 en 6 mois)", "Une part de l'écart est du risque réel (contrats courts, ancienneté)"],
    ["<b>3. Pilote « jeune actif »</b> : montant plafonné, et quelques refus tirés au sort acceptés à l'essai",
     f"Jeunes accordés : {rembourse_j:.0%} remboursés sans incident ; le sort des refusés n'est jamais observé",
     "Incidents de paiement de la cohorte pilote : moins de 5 %", f"Petit échantillon ({len(jeunes_acc)} jeunes accordés)"],
    ["<b>4. Suivi d'équité chaque trimestre</b> et dossier écrit (usage à haut risque, IA Act)",
     f"Ratio de parité d'âge passé de {ratio_av:.2f} à {ratio_ap:.2f} après le score V2",
     "Ratios de parité âge, genre, région ; alerte sous 0,8", "Groupes de moins de 300 demandes : chiffres fragiles"],
    ["<b>5. Comprendre pourquoi les 65 ans et plus sont peu clients</b> (agences, courtiers locaux)",
     f"65 ans et plus : {part65:.0%} des clients contre 26 % de la population, mais {tx65:.0%} de leurs demandes acceptées",
     "Part des 65 ans et plus et de l'Outre-mer parmi les clients", "Référence nationale, pas celle des demandeurs de crédit"],
])
esg = ('<section class="ok"><h2>Au-delà du chiffre d\'affaires</h2><ul style="font-size:16px;line-height:1.6;margin:6px 0">'
       '<li><b>Social :</b> ratios de parité et accès au crédit pour tous les âges.</li>'
       '<li><b>Gouvernance :</b> relecture humaine, dossier de conformité, revue du score.</li>'
       '<li><b>Environnement :</b> non mesurable avec ces données ; à collecter (part de rénovation énergétique dans les crédits travaux).</li>'
       '<li><b>Retours utilisateurs :</b> plaintes des jeunes actifs à recueillir pour confirmer ou nuancer les chiffres.</li></ul></section>')
decision = ('<section class="ok"><h2>Trois décisions attendues</h2><ol style="font-size:18px;line-height:1.7;margin:6px 0">'
            '<li>Feu vert pour tester un seuil ajusté pendant deux trimestres.</li>'
            '<li>Audit du score pour les moins de 35 ans, avec relecture humaine.</li>'
            '<li>Suivi d\'équité chaque trimestre, avec alerte sous 0,8.</li></ol>'
            f'<p style="font-size:16px">Enjeu : le nouveau score coûte environ <b>{fmt(m_av - m_ap)} € de PNB par mois</b>.</p></section>')

blocs = [
    nav,
    etape("1", "contexte", "Le contexte", "Que s'est-il passé chez Crédit Azur, pour qui, et que faire ?"), contexte,
    etape("2", "tendance", "Ce que disent les données", "Combien la banque gagne-t-elle, et quand cela a-t-il changé ?"),
    ("Chiffres clés", kpis, "Trois indicateurs : PNB estimé 2025 et évolution face à 2024, part du type de crédit principal, taux d'accord 2025."),
    ("Tendance", courbe, "Courbe du PNB mensuel avec sa tendance sur 12 mois et la date du déploiement du score V2 : le niveau baisse durablement après cette date."),
    etape("3", "segments", "Les segments qui comptent", "Quels crédits font vivre la banque, et la baisse les touche-t-elle tous ?"),
    ("PNB par type de crédit", detail, "Courbes mensuelles du PNB pour chaque type de crédit, avec un sélecteur pour isoler un type."),
    etape("4", "biais", "Les biais détectés et leurs limites", "Qui obtient un crédit, qui est absent des clients, et depuis quand ?"),
    alertes,
    ("Représentativité des clients", fig_rep, "Barres du ratio part observée sur part de référence par groupe, avec les seuils 0,8 et 1,25 ; le statut est écrit sur chaque barre."),
    ("Parité de la décision", parite, "Taux de crédit accordé par groupe, avec l'effectif, et la ligne des quatre cinquièmes du meilleur taux ; les groupes en dessous sont signalés par le texte « sous 4/5 »."),
    etape("5", "recommandations", "Les recommandations", "Que faire, comment le suivre, et avec quelles limites ?"), reco, esg,
    etape("6", "decision", "La décision attendue", "Qu'attendons-nous de vous aujourd'hui ?"), decision,
]
taille = exporter_html(f"{cfg['nom']} · tableau de bord", blocs, f"dashboard_{SECTEUR}.html",
                       sous_titre=f"Score V2 et accès au crédit · {df[cfg['date']].min():%m/%Y} → {df[cfg['date']].max():%m/%Y}")
print(f"dashboard_{SECTEUR}.html : {taille:.1f} Mo")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(f"dashboard_{SECTEUR}.html")   # le fichier arrive dans vos téléchargements


dashboard_banque.html : 4.9 Mo


## ✅ Check-list du livrable
- [ ] Le fichier s'ouvre seul dans un navigateur, sans connexion.
- [ ] Chaque graphique a un titre qui conclut et une description.
- [ ] Les alertes sont écrites, pas seulement colorées.
- [ ] Les couleurs restent lisibles pour un daltonien (palette Okabe-Ito).

💾 Enregistrez le notebook dans GitHub (« Dashboard Plotly exporté ») et envoyez le fichier HTML téléchargé au formateur.